# Kiểm tra & Thống kê Bất thường trên toàn bộ Microsoft GeoLife
Notebook này dùng để quét nhanh toàn bộ 182 user nhằm:
1. Tìm các file thực sự chứa mã lỗi `altitude = -777`.
2. Kiểm tra danh sách user có file gán nhãn `labels.txt`.
3. Lấy đường dẫn file mẫu bị lỗi để đưa về `01_eda_geolife.ipynb` kiểm thử.

In [1]:
import glob
import os

DATA_DIR = "..\data\Geolife Trajectories 1.3\Data"

# Lấy danh sách toàn bộ file .plt
all_plt_files = glob.glob(f"{DATA_DIR}/*/Trajectory/*.plt")
print(f"Tổng số file .plt tìm thấy: {len(all_plt_files)}")

files_with_777 = []

# Giới hạn tìm tối đa 5 file mẫu để dừng sớm
MAX_SAMPLES = 5

for file_path in all_plt_files:
    with open(file_path, "r", encoding="utf-8", errors="ignore") as f:
        # Bỏ qua 6 dòng metadata
        for _ in range(6):
            next(f, None)
        
        for line in f:
            parts = line.strip().split(",")
            # Cột index 3 là Altitude
            if len(parts) > 3 and parts[3].strip() == "-777":
                files_with_777.append(file_path)
                break
                
    if len(files_with_777) >= MAX_SAMPLES:
        break

print(f"\n✅ Đã tìm thấy {len(files_with_777)} file mẫu chứa -777:")
for path in files_with_777:
    print(f"-> {path}")

Tổng số file .plt tìm thấy: 18670

✅ Đã tìm thấy 5 file mẫu chứa -777:
-> ..\data\Geolife Trajectories 1.3\Data\002\Trajectory\20090216200646.plt
-> ..\data\Geolife Trajectories 1.3\Data\005\Trajectory\20081123144433.plt
-> ..\data\Geolife Trajectories 1.3\Data\010\Trajectory\20080328144824.plt
-> ..\data\Geolife Trajectories 1.3\Data\010\Trajectory\20080328160001.plt
-> ..\data\Geolife Trajectories 1.3\Data\010\Trajectory\20080329160048.plt


In [2]:
import pandas as pd

if files_with_777:
    target_file = files_with_777[0]
    print(f"Đang kiểm tra file: {target_file}")
    
    columns = ["lat", "lon", "reserved", "altitude", "date_days", "date_str", "time_str"]
    df_check = pd.read_csv(target_file, skiprows=6, header=None, names=columns)
    
    # Lọc ra các dòng có altitude = -777
    error_rows = df_check[df_check["altitude"] == -777]
    print(f"\nTổng số điểm GPS trong file: {len(df_check)}")
    print(f"Số điểm có altitude = -777: {len(error_rows)} ({len(error_rows)/len(df_check)*100:.2f}%)")
    
    # Hiển thị 5 dòng bị lỗi đầu tiên
    display(error_rows.head())
else:
    print("Chưa tìm thấy file lỗi. Hãy kiểm tra lại đường dẫn DATA_DIR.")

Đang kiểm tra file: ..\data\Geolife Trajectories 1.3\Data\002\Trajectory\20090216200646.plt

Tổng số điểm GPS trong file: 679
Số điểm có altitude = -777: 1 (0.15%)


,lat,lon,reserved,altitude,date_days,date_str,time_str
593,39.897409,116.38381,0,-777,39861.014549,2009-02-17,00:20:57


In [3]:
label_files = glob.glob(f"{DATA_DIR}/*/labels.txt")

labeled_users = [os.path.basename(os.path.dirname(p)) for p in label_files]
print(f"Số lượng user có file labels.txt: {len(labeled_users)} / 182")
print(f"Danh sách user có nhãn: {labeled_users}")

Số lượng user có file labels.txt: 69 / 182
Danh sách user có nhãn: ['010', '020', '021', '052', '053', '056', '058', '059', '060', '062', '064', '065', '067', '068', '069', '073', '075', '076', '078', '080', '081', '082', '084', '085', '086', '087', '088', '089', '091', '092', '096', '097', '098', '100', '101', '102', '104', '105', '106', '107', '108', '110', '111', '112', '114', '115', '116', '117', '118', '124', '125', '126', '128', '129', '136', '138', '139', '141', '144', '147', '153', '154', '161', '163', '167', '170', '174', '175', '179']


## Vấn đề 3 trong Guide: GPS Drift & Nhiễu vận tốc cực đoan (Speed Outliers)

**Mô tả vấn đề trong User Guide / Paper:**
* **Nguyên nhân vật lý (Urban Canyon & Multipath Effect):** Khi người dùng đi vào giữa các tòa nhà cao tầng, đường hầm, hoặc đứng đợi đèn đỏ, tín hiệu vệ tinh bị phản xạ khiến tọa độ GPS bị "nhảy cóc" liên tục trong một phạm vi nhỏ.
* **Hậu quả:** 
  * Tạo ra vận tốc tức thời ảo lên đến hàng trăm km/h dù người dùng đang đứng yên hoặc đi bộ.
  * Tích lũy quãng đường di chuyển ảo (quỹ đạo bị rung/răng cưa), làm sai lệch toàn bộ đặc trưng chuyển động.

**Mục tiêu của Cell này:**
1. Áp dụng công thức Haversine vectorized để tính khoảng cách giữa 2 điểm liên tiếp.
2. Lấy mẫu thử nghiệm trên 100 file ngẫu nhiên để thống kê: **Có bao nhiêu điểm GPS vượt ngưỡng vận tốc thực tế ($v > 180\text{ km/h}$)?**
3. Đánh giá tỷ lệ phần trăm nhiễu trên toàn tập dữ liệu để thiết lập ngưỡng lọc (Threshold Filtering) cho bước làm sạch.

In [4]:
import random
import numpy as np
import pandas as pd

def fast_haversine(lat1, lon1, lat2, lon2):
    """Tính khoảng cách bề mặt Trái Đất (mét) giữa 2 mảng tọa độ."""
    R = 6371000
    p1, p2 = np.radians(lat1), np.radians(lat2)
    dp, dl = np.radians(lat2 - lat1), np.radians(lon2 - lon1)
    a = np.sin(dp / 2.0)**2 + np.cos(p1) * np.cos(p2) * np.sin(dl / 2.0)**2
    return 2 * R * np.arctan2(np.sqrt(a), np.sqrt(1 - a))

# Lấy ngẫu nhiên 100 file để audit tốc độ lấy mẫu
random.seed(42)
sample_files = random.sample(all_plt_files, min(100, len(all_plt_files)))

drift_count = 0
total_checked_points = 0
SPEED_LIMIT_KMH = 180  # Ngưỡng vận tốc phi thực tế (trừ khi đi máy bay/tàu cao tốc)

for file_path in sample_files:
    # Chỉ đọc các cột: lat (0), lon (1), date_str (5), time_str (6)
    df_temp = pd.read_csv(
        file_path, 
        skiprows=6, 
        header=None, 
        usecols=[0, 1, 5, 6], 
        names=['lat', 'lon', 'd_str', 't_str']
    )
    
    # Ghép datetime
    df_temp['dt'] = pd.to_datetime(df_temp['d_str'] + ' ' + df_temp['t_str'])
    
    # Tính delta time (giây) và delta distance (mét)
    delta_s = (df_temp['dt'] - df_temp['dt'].shift(1)).dt.total_seconds()
    dist_m = fast_haversine(
        df_temp['lat'].shift(1), df_temp['lon'].shift(1), 
        df_temp['lat'], df_temp['lon']
    )
    
    # Tính vận tốc km/h
    speed_kmh = (dist_m / delta_s.replace(0, np.nan)) * 3.6
    
    # Đếm số điểm vượt ngưỡng vận tốc
    drift_count += (speed_kmh > SPEED_LIMIT_KMH).sum()
    total_checked_points += len(df_temp)

print(f"--- KẾT QUẢ KIỂM TRA GPS DRIFT ---")
print(f"Số files kiểm tra mẫu: {len(sample_files)}")
print(f"Tổng số điểm GPS đã quét: {total_checked_points:,}")
print(f"Số điểm có vận tốc > {SPEED_LIMIT_KMH} km/h: {drift_count}")
print(f"Tỷ lệ điểm nhiễu: {(drift_count / total_checked_points) * 100:.4f}%")

--- KẾT QUẢ KIỂM TRA GPS DRIFT ---
Số files kiểm tra mẫu: 100
Tổng số điểm GPS đã quét: 128,739
Số điểm có vận tốc > 180 km/h: 106
Tỷ lệ điểm nhiễu: 0.0823%


## Vấn đề 4 trong Guide: Khoảng đứt gãy thời gian lớn & Cần phân đoạn quỹ đạo (Trajectory Segmentation)

**Mô tả vấn đề trong User Guide / Paper:**
* **Nguyên nhân:** Người dùng có thể tắt thiết bị thu GPS, thiết bị hết pin, hoặc đi vào vùng hoàn toàn mất sóng (như ga tàu điện ngầm) rồi vài tiếng sau mới bật lại. Nhiều bản ghi trong cùng một file `.plt` thực chất trải dài qua nhiều buổi hoặc nhiều ngày.
* **Hậu quả:** 
  * Nếu giữ nguyên và tính toán liên tục, khoảng cách giữa 2 điểm ghi nhận có thể lên tới 50 km với khoảng thời gian cách nhau 8 tiếng $\rightarrow$ Hệ thống sẽ hiểu nhầm đây là "một chặng di chuyển liên tục kéo dài 8 tiếng".
* **Giải pháp chuẩn của Microsoft Research:**
  * Đặt một ngưỡng thời gian tối đa $\Delta t_{threshold} = 20\text{ phút}$ ($1200\text{s}$). 
  * Nếu khoảng cách giữa hai điểm liên tiếp vượt quá $1200\text{s}$, bắt buộc phải **cắt (split)** file thành 2 quỹ đạo độc lập.

**Mục tiêu của Cell này:**
1. Quét một tập con các file `.plt` để đo lường: **Bao nhiêu phần trăm file chứa khoảng đứt quãng lớn hơn 20 phút?**
2. Khẳng định sự cần thiết phải có bước `Trajectory Segmentation` trong pipeline tiền xử lý.

In [5]:
# Ngưỡng thời gian gián đoạn theo tài liệu kỹ thuật của GeoLife: 20 phút = 1200 giây
TIME_GAP_THRESHOLD_SECONDS = 1200

files_with_large_gap = 0
checked_gap_files = sample_files[:50]  # Kiểm tra trên 50 file mẫu
gap_details = []

for file_path in checked_gap_files:
    df_temp = pd.read_csv(
        file_path, 
        skiprows=6, 
        header=None, 
        usecols=[5, 6], 
        names=['d_str', 't_str']
    )
    
    dt = pd.to_datetime(df_temp['d_str'] + ' ' + df_temp['t_str'])
    delta_s = (dt - dt.shift(1)).dt.total_seconds()
    
    # Tìm khoảng cách thời gian lớn nhất trong file
    max_gap = delta_s.max()
    
    if max_gap > TIME_GAP_THRESHOLD_SECONDS:
        files_with_large_gap += 1
        gap_details.append({
            "file": os.path.basename(file_path),
            "max_gap_minutes": round(max_gap / 60, 2)
        })

print(f"--- KẾT QUẢ KIỂM TRA ĐỨT GÃY THỜI GIAN (TIME GAP) ---")
print(f"Số files kiểm tra: {len(checked_gap_files)}")
print(f"Số files chứa khoảng trống > 20 phút: {files_with_large_gap} ({files_with_large_gap / len(checked_gap_files) * 100:.1f}%)")

print("\nVí dụ một số file có gap thời gian lớn nhất:")
df_gap_preview = pd.DataFrame(gap_details).sort_values(by="max_gap_minutes", ascending=False).head(5)
display(df_gap_preview)

--- KẾT QUẢ KIỂM TRA ĐỨT GÃY THỜI GIAN (TIME GAP) ---
Số files kiểm tra: 50
Số files chứa khoảng trống > 20 phút: 14 (28.0%)

Ví dụ một số file có gap thời gian lớn nhất:


,file,max_gap_minutes
4,20090412005338.plt,291.63
2,20090420103349.plt,222.13
1,20090115212022.plt,221.58
3,20090526023634.plt,218.18
7,20081230111834.plt,182.85


## Vấn đề 7: Trùng lặp Mốc thời gian (Duplicate Timestamps)

**Mô tả vấn đề:**
* **Bản chất kỹ thuật:** Dữ liệu GPS là chuỗi thời gian đơn điệu tăng dần ($t_1 < t_2 < t_3$). Tuy nhiên, do lỗi buffer của thiết bị ghi log hoặc quá trình nối file thô, có thể xuất hiện 2 hoặc nhiều dòng dữ liệu có cùng chính xác một mốc thời gian (`datetime`).
* **Hậu quả nghiêm trọng:** 
  * Khi tính chênh lệch thời gian: $\Delta t = t_i - t_{i-1} = 0\text{ giây}$.
  * Công thức tính vận tốc: $v = \frac{\Delta d}{\Delta t} = \frac{\Delta d}{0} \rightarrow \infty$ (Lỗi phép chia cho 0 - `ZeroDivisionError` hoặc sinh ra giá trị `Inf`).
  * Làm hỏng các mô hình chuỗi thời gian (RNN, LSTM, Transformer) vì các mô hình này giả định mỗi bước thời gian $t$ là duy nhất.

**Mục tiêu của Cell:**
1. Quét mẫu một nhóm file trên toàn bộ dataset để kiểm tra: **Có hiện tượng trùng lặp timestamp trong cùng 1 file không?**
2. Trích xuất và hiển thị trực tiếp các cặp dòng bị trùng để xác định: Trùng cả tọa độ (hoàn toàn duplicate) hay cùng giờ nhưng khác tọa độ (xung đột dữ liệu).
3. Đề xuất quy tắc xử lý: `drop_duplicates(subset=['datetime'])` giữ lại điểm đầu hay lấy trung bình.

In [6]:
import pandas as pd
import random

# Lấy ngẫu nhiên 50 file để kiểm tra trùng lặp
random.seed(42)
sample_dup_files = random.sample(all_plt_files, min(50, len(all_plt_files)))

files_with_duplicates = []
total_duplicate_rows = 0

print("🔍 Đang quét kiểm tra trùng lặp timestamp...")

for file_path in sample_dup_files:
    # Chỉ đọc cột date_str (5) và time_str (6) để tối ưu tốc độ
    df_temp = pd.read_csv(
        file_path, 
        skiprows=6, 
        header=None, 
        usecols=[0, 1, 5, 6], 
        names=['lat', 'lon', 'd_str', 't_str']
    )
    
    df_temp['datetime'] = pd.to_datetime(df_temp['d_str'] + ' ' + df_temp['t_str'])
    
    # Kiểm tra số lượng dòng trùng timestamp
    dup_count = df_temp.duplicated(subset=['datetime']).sum()
    
    if dup_count > 0:
        files_with_duplicates.append((file_path, dup_count))
        total_duplicate_rows += dup_count

print(f"\n--- KẾT QUẢ KIỂM TRA TRÙNG LẶP TIMESTAMP ---")
print(f"Số files quét mẫu: {len(sample_dup_files)}")
print(f"Số files phát hiện có trùng lặp: {len(files_with_duplicates)}")
print(f"Tổng số bản ghi bị trùng: {total_duplicate_rows}")

# Hiển thị chi tiết ví dụ từ file đầu tiên bị lỗi (nếu có)
if files_with_duplicates:
    target_file, count = files_with_duplicates[0]
    print(f"\n👉 Chi tiết ví dụ từ file: {target_file} (Chứa {count} dòng trùng):")
    
    columns = ["lat", "lon", "reserved", "altitude", "date_days", "date_str", "time_str"]
    df_detail = pd.read_csv(target_file, skiprows=6, header=None, names=columns)
    df_detail['datetime'] = pd.to_datetime(df_detail['date_str'] + ' ' + df_detail['time_str'])
    
    # Lấy TẤT CẢ các dòng tham gia vào sự trùng lặp (keep=False) để đối chiếu
    dup_samples = df_detail[df_detail.duplicated(subset=['datetime'], keep=False)]
    display(dup_samples[['datetime', 'lat', 'lon', 'altitude']].head(10))
else:
    print("\n✅ Trong 50 file mẫu kiểm tra không xuất hiện trùng lặp timestamp.")

🔍 Đang quét kiểm tra trùng lặp timestamp...

--- KẾT QUẢ KIỂM TRA TRÙNG LẶP TIMESTAMP ---
Số files quét mẫu: 50
Số files phát hiện có trùng lặp: 2
Tổng số bản ghi bị trùng: 4

👉 Chi tiết ví dụ từ file: ..\data\Geolife Trajectories 1.3\Data\010\Trajectory\20071231170243.plt (Chứa 3 dòng trùng):


,datetime,lat,lon,altitude
37,2007-12-31 17:03:44,30.248485,120.157200,66
38,2007-12-31 17:03:44,30.248492,120.157100,66
39,2007-12-31 17:03:46,30.248448,120.157383,72
40,2007-12-31 17:03:46,30.248463,120.157290,71
83,2007-12-31 17:04:38,30.248268,120.159430,67
84,2007-12-31 17:04:38,30.248283,120.159370,66


## Vấn đề 8: Kiểm tra Giới hạn Tọa độ & Đơn vị Độ cao (Feet vs. Meters)

**Mô tả vấn đề trong User Guide:**
1. **Đơn vị Độ cao:** Cột 4 trong file `.plt` được lưu trữ dưới đơn vị **Feet**, không phải Mét theo chuẩn SI. Khi tính toán các đặc trưng địa hình cần nhân hệ số `0.3048`.
2. **Biên độ Tọa độ (Bounds Check):** 
   - Tọa độ hợp lệ vật lý: $\text{Lat} \in [-90, 90]$ và $\text{Lon} \in [-180, 180]$.
   - Cần kiểm tra xem có xuất hiện điểm $(0.0, 0.0)$ (lỗi mất định vị thường gặp ở chip GPS) hoặc tọa độ ngoài phạm vi không.

In [7]:
invalid_coords = 0
total_pts_checked = 0
altitudes = []

# Quét trên tập 50 file mẫu
for file_path in sample_files[:50]:
    df_geo = pd.read_csv(
        file_path, 
        skiprows=6, 
        header=None, 
        usecols=[0, 1, 3], 
        names=['lat', 'lon', 'altitude_feet']
    )
    
    # Loại trừ mã lỗi -777 trước khi thống kê độ cao
    valid_alt = df_geo.loc[df_geo['altitude_feet'] != -777, 'altitude_feet'] * 0.3048 # Đổi sang mét
    altitudes.extend(valid_alt.tolist())
    
    # Kiểm tra tọa độ ngoài biên vật lý hoặc tọa độ (0, 0)
    out_of_bounds = (
        (df_geo['lat'] < -90) | (df_geo['lat'] > 90) |
        (df_geo['lon'] < -180) | (df_geo['lon'] > 180) |
        ((df_geo['lat'] == 0) & (df_geo['lon'] == 0))
    )
    invalid_coords += out_of_bounds.sum()
    total_pts_checked += len(df_geo)

df_alt = pd.Series(altitudes)

print("--- KẾT QUẢ KIỂM TRA TỌA ĐỘ & ĐỘ CAO ---")
print(f"Tổng số điểm kiểm tra: {total_pts_checked:,}")
print(f"Số điểm có tọa độ ngoài biên hoặc (0,0): {invalid_coords}")
print("\nPhân phối độ cao sau khi đổi sang Mét:")
print(f" - Min altitude: {df_alt.min():.2f} m")
print(f" - Max altitude: {df_alt.max():.2f} m")
print(f" - Median altitude: {df_alt.median():.2f} m")

--- KẾT QUẢ KIỂM TRA TỌA ĐỘ & ĐỘ CAO ---
Tổng số điểm kiểm tra: 58,410
Số điểm có tọa độ ngoài biên hoặc (0,0): 0

Phân phối độ cao sau khi đổi sang Mét:
 - Min altitude: -1514.00 m
 - Max altitude: 7680.02 m
 - Median altitude: 48.16 m


## Vấn đề 8B: Quét tìm File có Tọa độ Bất thường / Xa vị trí chuẩn & Visualize Thực tế

**Mục tiêu:**
1. Quét tìm các file chứa điểm tọa độ lỗi vật lý:
   - Tọa độ ngoài dải $\text{Lat} \notin [-90, 90]$ hoặc $\text{Lon} \notin [-180, 180]$.
   - Tọa độ $(0.0, 0.0)$ (lỗi mất sóng GPS phổ biến - "Null Island").
   - Tọa độ nằm ngoài lãnh thổ Trung Quốc (User Guide ghi nhận có các chuyến đi sang Châu Âu/Mỹ).
2. Lấy đường dẫn file đại diện và trực quan hóa lên bản đồ Folium với góc nhìn toàn cầu.

In [11]:
import os
import glob
import pandas as pd

# ==================== KHAI BÁO ĐƯỜNG DẪN DỮ LIỆU TẠI ĐÂY ====================
# Dùng tiền tố r để tránh lỗi escape sequence của dấu gạch chéo ngược trên Windows
DATA_DIR = r"..\data\Geolife Trajectories 1.3\Data"

# Tìm toàn bộ danh sách file .plt từ thư mục Data
all_plt_files = glob.glob(os.path.join(DATA_DIR, "*", "Trajectory", "*.plt"))
print(f"Tổng số file .plt tìm thấy trong thư mục: {len(all_plt_files):,}")
assert len(all_plt_files) > 0, "Không tìm thấy file nào! Vui lòng kiểm tra lại đường dẫn DATA_DIR."
# ============================================================================

print("🔍 Đang quét toàn bộ file để tìm tọa độ ngoài biên / nước ngoài...")

# Phạm vi xấp xỉ của Trung Quốc: Lat ~ [18, 54], Lon ~ [73, 135]
# Phạm vi đô thị Bắc Kinh: Lat ~ [39.4, 41.1], Lon ~ [115.4, 117.5]

files_invalid_physical = []   # Lỗi ngoài dải Trái Đất hoặc (0,0)
files_outside_china = []      # Điểm ở nước ngoài (Mỹ, Châu Âu...)

# Quét trên tập all_plt_files
for file_path in all_plt_files:
    df_coords = pd.read_csv(
        file_path, 
        skiprows=6, 
        header=None, 
        usecols=[0, 1], 
        names=['lat', 'lon']
    )
    
    # 1. Kiểm tra lỗi vật lý hoặc điểm (0, 0)
    is_invalid = (
        (df_coords['lat'] < -90) | (df_coords['lat'] > 90) |
        (df_coords['lon'] < -180) | (df_coords['lon'] > 180) |
        ((df_coords['lat'] == 0) & (df_coords['lon'] == 0))
    )
    if is_invalid.any():
        files_invalid_physical.append(file_path)
        continue

    # 2. Kiểm tra các chuyến đi ngoài Trung Quốc
    is_outside_china = (
        (df_coords['lat'] < 18) | (df_coords['lat'] > 54) |
        (df_coords['lon'] < 73) | (df_coords['lon'] > 135)
    )
    if is_outside_china.any():
        files_outside_china.append(file_path)
        
    if len(files_outside_china) >= 5 and len(files_invalid_physical) >= 1:
        break

print("\n--- KẾT QUẢ QUÉT TỌA ĐỘ ---")
print(f"Số file chứa lỗi vật lý / (0,0): {len(files_invalid_physical)}")
print(f"Số file tìm thấy có chuyến đi ở nước ngoài: {len(files_outside_china)}")

# ==================== HIỂN THỊ DẠNG BẢNG (TABLE VISUALIZATION) ====================

columns = ["lat", "lon", "reserved", "altitude", "date_days", "date_str", "time_str"]

# 1. Bảng hiển thị lỗi vật lý / (0,0) nếu có
if files_invalid_physical:
    err_file = files_invalid_physical[0]
    print(f"\n👉 BẢNG CHI TIẾT DÒNG LỖI VẬT LÝ HOẶC (0,0): {os.path.basename(err_file)}")
    print(f"Đường dẫn file: {err_file}")
    df_err = pd.read_csv(err_file, skiprows=6, header=None, names=columns)
    
    cond_phys = (
        (df_err['lat'] < -90) | (df_err['lat'] > 90) |
        (df_err['lon'] < -180) | (df_err['lon'] > 180) |
        ((df_err['lat'] == 0) & (df_err['lon'] == 0))
    )
    display(df_err[cond_phys].head(10))
else:
    print("\n✅ Không phát hiện dòng nào vi phạm biên độ vật lý [-90, 90], [-180, 180] hoặc (0,0).")

# 2. Bảng hiển thị các điểm tọa độ ở nước ngoài (ngoài Trung Quốc)
if files_outside_china:
    intl_file = files_outside_china[0]
    print(f"\n👉 BẢNG MẪU TỌA ĐỘ NGOÀI LÃNH THỔ TRUNG QUỐC: {os.path.basename(intl_file)}")
    print(f"Đường dẫn file: {intl_file}")
    df_intl = pd.read_csv(intl_file, skiprows=6, header=None, names=columns)
    
    cond_intl = (
        (df_intl['lat'] < 18) | (df_intl['lat'] > 54) |
        (df_intl['lon'] < 73) | (df_intl['lon'] > 135)
    )
    
    # Hiển thị 5 dòng đầu ngoài Trung Quốc cùng thống kê min/max
    display(df_intl[cond_intl][['date_str', 'time_str', 'lat', 'lon', 'altitude']].head(5))
    
    # Thống kê ngắn gọn phạm vi tọa độ của file này
    summary_intl = pd.DataFrame({
        "Metric": ["Min Latitude", "Max Latitude", "Min Longitude", "Max Longitude"],
        "Value": [df_intl['lat'].min(), df_intl['lat'].max(), df_intl['lon'].min(), df_intl['lon'].max()]
    })
    print("Phạm vi tọa độ của file quốc tế mẫu:")
    display(summary_intl)

Tổng số file .plt tìm thấy trong thư mục: 18,670
🔍 Đang quét toàn bộ file để tìm tọa độ ngoài biên / nước ngoài...

--- KẾT QUẢ QUÉT TỌA ĐỘ ---
Số file chứa lỗi vật lý / (0,0): 1
Số file tìm thấy có chuyến đi ở nước ngoài: 5

👉 BẢNG CHI TIẾT DÒNG LỖI VẬT LÝ HOẶC (0,0): 20110911000506.plt
Đường dẫn file: ..\data\Geolife Trajectories 1.3\Data\020\Trajectory\20110911000506.plt


,lat,lon,reserved,altitude,date_days,date_str,time_str
6775,400.166667,116.21539,0,0,40797.141597,2011-09-11,03:23:54



👉 BẢNG MẪU TỌA ĐỘ NGOÀI LÃNH THỔ TRUNG QUỐC: 20081117000642.plt
Đường dẫn file: ..\data\Geolife Trajectories 1.3\Data\017\Trajectory\20081117000642.plt


,date_str,time_str,lat,lon,altitude
0,2008-11-17,00:06:42,24.471738,142.879508,-446.2
1,2008-11-17,00:06:43,24.471483,142.879457,-446.2
2,2008-11-17,00:06:44,24.471227,142.879438,-446.2
3,2008-11-17,00:06:45,24.470932,142.879482,-446.2
4,2008-11-17,00:06:46,24.470680,142.879495,-442.9


Phạm vi tọa độ của file quốc tế mẫu:


,Metric,Value
0,Min Latitude,24.468977
1,Max Latitude,24.471738
2,Min Longitude,142.879438
3,Max Longitude,142.879643


In [ ]:
import os
import folium
import pandas as pd
from IPython.display import IFrame

columns = ["lat", "lon", "reserved", "altitude", "date_days", "date_str", "time_str"]
exported_maps = []

# 1. Vẽ và xuất file HTML cho các quỹ đạo lỗi vật lý (Physical Errors)
for idx, file_path in enumerate(files_invalid_physical):
    fname = os.path.basename(file_path)
    df_p = pd.read_csv(file_path, skiprows=6, header=None, names=columns)
    
    # Lọc điểm hợp lệ và bắt các điểm lỗi
    invalid_mask = (df_p['lat'] < -90) | (df_p['lat'] > 90) | (df_p['lon'] < -180) | (df_p['lon'] > 180)
    invalid_count = invalid_mask.sum()
    
    df_valid = df_p[~invalid_mask]
    coords_valid = df_valid[['lat', 'lon']].values.tolist()
    
    if not coords_valid:
        continue
        
    center_lat = df_valid['lat'].median()
    center_lon = df_valid['lon'].median()
    
    # Khởi tạo map
    m = folium.Map(location=[center_lat, center_lon], zoom_start=12, tiles="OpenStreetMap")
    
    # Vẽ đường đi hợp lệ (Màu đỏ cảnh báo)
    folium.PolyLine(coords_valid, color="red", weight=3.5, opacity=0.85, tooltip=f"Lỗi: {fname}").add_to(m)
    folium.Marker(coords_valid[0], popup=f"Start [{fname}]", icon=folium.Icon(color="darkred", icon="play")).add_to(m)
    folium.Marker(coords_valid[-1], popup=f"End [{fname}]", icon=folium.Icon(color="darkred", icon="stop")).add_to(m)
    
    # Lưu file HTML riêng biệt
    html_name = f"map_error_physical_{idx+1}_{fname.replace('.plt', '')}.html"
    m.save(html_name)
    exported_maps.append({
        "type": "Lỗi vật lý",
        "file_plt": fname,
        "html_file": html_name,
        "note": f"Chứa {invalid_count} điểm ngoài biên (ví dụ lat=400)"
    })

# 2. Vẽ và xuất từng file HTML cho các chuyến đi nước ngoài (International Trajectories)
palette = ["blue", "purple", "darkgreen", "orange", "cadetblue"]

for idx, file_path in enumerate(files_outside_china):
    fname = os.path.basename(file_path)
    df_intl = pd.read_csv(file_path, skiprows=6, header=None, names=columns)
    
    # Lọc biên độ an toàn
    valid_intl = df_intl[
        (df_intl['lat'] >= -90) & (df_intl['lat'] <= 90) & 
        (df_intl['lon'] >= -180) & (df_intl['lon'] <= 180)
    ]
    coords_intl = valid_intl[['lat', 'lon']].values.tolist()
    
    if not coords_intl:
        continue
        
    center_lat = valid_intl['lat'].median()
    center_lon = valid_intl['lon'].median()
    
    # Khởi tạo map
    m = folium.Map(location=[center_lat, center_lon], zoom_start=6, tiles="OpenStreetMap")
    
    color = palette[idx % len(palette)]
    folium.PolyLine(coords_intl, color=color, weight=3.5, opacity=0.85, tooltip=f"Quốc tế: {fname}").add_to(m)
    folium.Marker(coords_intl[0], popup=f"Start [{fname}]", icon=folium.Icon(color="green", icon="play")).add_to(m)
    folium.Marker(coords_intl[-1], popup=f"End [{fname}]", icon=folium.Icon(color="gray", icon="stop")).add_to(m)
    
    # Lưu file HTML riêng biệt
    html_name = f"map_intl_{idx+1}_{fname.replace('.plt', '')}.html"
    m.save(html_name)
    exported_maps.append({
        "type": "Quốc tế ngoài TQ",
        "file_plt": fname,
        "html_file": html_name,
        "note": f"Lat: [{valid_intl['lat'].min():.2f}, {valid_intl['lat'].max():.2f}], Lon: [{valid_intl['lon'].min():.2f}, {valid_intl['lon'].max():.2f}]"
    })

# ==================== HIỂN THỊ DANH SÁCH FILE HTML VỪA XUẤT ====================
df_exported = pd.DataFrame(exported_maps)
print("✅ ĐÃ XUẤT THÀNH CÔNG TỪNG FILE HTML RIÊNG BIỆT:")
display(df_exported[['type', 'file_plt', 'html_file', 'note']])

✅ ĐÃ XUẤT THÀNH CÔNG TỪNG FILE HTML RIÊNG BIỆT:


,type,file_plt,html_file,note
0,Lỗi vật lý,20110911000506.plt,map_error_physical_1_20110911000506.html,Chứa 1 điểm ngoài biên (ví dụ lat=400)
1,Quốc tế ngoài TQ,20081117000642.plt,map_intl_1_20081117000642.html,"Lat: [24.47, 24.47], Lon: [142.88, 142.88]"
2,Quốc tế ngoài TQ,20070614052022.plt,map_intl_2_20070614052022.html,"Lat: [34.99, 35.01], Lon: [135.76, 135.77]"
3,Quốc tế ngoài TQ,20070615033934.plt,map_intl_3_20070615033934.html,"Lat: [35.01, 35.01], Lon: [135.75, 135.75]"
4,Quốc tế ngoài TQ,20080801200436.plt,map_intl_4_20080801200436.html,"Lat: [47.63, 47.63], Lon: [-122.14, -122.14]"
5,Quốc tế ngoài TQ,20080802182334.plt,map_intl_5_20080802182334.html,"Lat: [47.63, 47.63], Lon: [-122.14, -122.14]"



👉 Đang hiển thị trực tiếp file: map_error_physical_1_20110911000506.html
